<p align="center">
    <img src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Vertical-SinFondo.png" width="500"/>
</p>

<h2 align="center"><i>ITESO, Universidad Jesuita de Guadalajara</i></h2>
<h2 align="center"><i>Quantitative Finance</i></h2>
<h2 align="center"><i>Prof. Luis Carlos Alvarado Garnica</i></h2>

# Heston — Las Griegas
---
En clase derivamos a mano que $\Delta = P_1$ explotando que $S_t$ entra a $\varphi_j$ **linealmente y sola** en el exponente. Este notebook hace dos cosas:

1. **Demuestra simbólicamente con `sympy`** la cancelación algebraica del $iu$ que hace posible esa derivación — el mismo paso que vimos en el pizarrón, ahora verificado por código.
2. **Verifica numéricamente** que Delta por diferencias finitas coincide con $P_1$ calculado por Fourier.
3. Entrega un **motor genérico de diferencias finitas** para que ustedes calculen el resto de las griegas sin tener que rederivar cada una a mano, exactamente como se hace en la práctica.


## 0. Librerias

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.integrate import quad
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f9f9f9',
                     'axes.grid':True,'grid.alpha':0.35,'font.size':11,'lines.linewidth':2})
PURPLE='#534AB7'; GREEN='#1D9E75'; ORANGE='#D85A30'; AMBER='#BA7517'; BLUE='#2A7FBF'


## 1. Motor de pricing

Las mismas funciones de siempre: función característica en forma estable e inversión de Fourier para $P_1,P_2$ y el precio del call.


In [ ]:
def heston_cf_j(u, j, x, v, tau, r, kappa, theta, xi, rho):
    i=1j; uj=0.5 if j==1 else -0.5; bj=kappa-rho*xi if j==1 else kappa
    dj=np.sqrt((rho*xi*i*u-bj)**2+xi**2*(u**2-2*uj*i*u))
    g2j=(bj-rho*xi*i*u+dj)/(bj-rho*xi*i*u-dj)
    Dj=((bj-rho*xi*i*u+dj)/xi**2)*((1-np.exp(dj*tau))/(1-g2j*np.exp(dj*tau)))
    Cj=r*i*u*tau+(kappa*theta/xi**2)*((bj-rho*xi*i*u+dj)*tau-2*np.log((1-g2j*np.exp(dj*tau))/(1-g2j)))
    return np.exp(Cj+Dj*v+i*u*x)

def heston_prob_j(j, x, v, tau, K, r, kappa, theta, xi, rho):
    lnK = np.log(K)
    integrand = lambda u: np.real(np.exp(-1j*u*lnK)*heston_cf_j(u,j,x,v,tau,r,kappa,theta,xi,rho)/(1j*u))
    val, _ = quad(integrand, 1e-8, 200, limit=200)
    return 0.5 + val/np.pi

def heston_call(S, K, r, tau, v0, kappa, theta, xi, rho):
    x = np.log(S)
    P1 = heston_prob_j(1, x, v0, tau, K, r, kappa, theta, xi, rho)
    P2 = heston_prob_j(2, x, v0, tau, K, r, kappa, theta, xi, rho)
    return S*P1 - K*np.exp(-r*tau)*P2


## 2. Verificación numérica: Delta (diferencias finitas) $\approx$ $P_1$ (Fourier)

Con la cancelación demostrada simbólicamente, verificamos que en la práctica $\Delta = \partial C/\partial S_t$, calculado por diferencias finitas centradas sobre el precio, coincide con $P_1$ calculado directamente por la integral de Fourier, sin pasar por ninguna diferencia finita.


In [ ]:
S0, K, r, tau = 100.0, 100.0, 0.05, 1.0
v0, kappa, theta, xi, rho = 0.04, 2.0, 0.04, 0.30, -0.7      
h = 1e-3

# Código aquí


# print(f"Delta (diferencias finitas sobre el precio): {delta_fd:.6f}")
# print(f"P1    (integral de Fourier, analitico):       {P1:.6f}")
# print(f"Diferencia:                                    {abs(delta_fd-P1):.2e}")
# print()
# print("Coinciden hasta el error de la cuadratura numerica -> Delta = P1, confirmado en codigo.")


## 3. El resto de las griegas: motor genérico de diferencias finitas

Como vimos en la presentación, $\kappa,\theta,\xi,\rho$ están enredados dentro de $d_j$ y $g_{2j}$ — no hay una cancelación limpia como la de Delta. La práctica estándar (y lo que van a usar de aqui en adelante) es un **motor genérico**: perturbar cada parámetro, re-priciar con el mismo motor de Heston, y dividir.

$$\frac{\partial C}{\partial G} \approx \frac{C(G+h) - C(G-h)}{2h}$$

Esta función calcula todas las griegas de una sola vez.


In [ ]:
def heston_greeks_fd(S0, K, r, tau, v0, kappa, theta, xi, rho, h_rel=1e-4):
   

    # return {
    #     'price': base, 'delta': delta, 'gamma': gamma,
    #     'vega_v0': vega_v0, 'vega_theta': vega_theta, 'vega_kappa': vega_kappa,
    #     'vega_xi': vega_xi, 'vega_rho': vega_rho,
    #     'theta': theta_g, 'rho': rho_g
    # }
    pass

griegas = heston_greeks_fd(S0, K, r, tau, v0, kappa, theta, xi, rho)
print(f"{'Griega':14}{'Valor':>12}")
print("-"*26)
for nombre, val in griegas.items():
    print(f"{nombre:14}{val:>12.6f}")


## 4. Comparación directa: Delta analítico vs Delta del motor genérico

Como cierre, confirmamos que el `delta` que sale del motor genérico de diferencias finitas  coincide con el `P1` analítico, es la misma cantidad, calculada por dos caminos distintos.


In [ ]:
print(f"Delta del motor generico (FD): {griegas['delta']:.6f}")
print(f"P1 analitico (Fourier):         {P1:.6f}")
print(f"Diferencia:                     {abs(griegas['delta']-P1):.2e}")
